# Chapter 3 &mdash; Start Here: Languages in Detail

**Concept 0 of the Chapter 3 decomposition:** *Start Here: the Language Toolkit, from Zero and Unit to Star*

The catch-all notebook for Chapters 2&ndash;3. It defines every language builder
Jove uses &mdash; `lphi`, `lunit`, `lcat`, `lexp`, `lstar`, `lrev`, `lhomo`,
`lcomplem`, ... &mdash; in full, and drives each one with small experiments. The
sixteen concept notebooks of this chapter then take one idea at a time; links to
them appear at the end of every section below.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter3-Star/Concept-START-HERE/Concept-START-HERE.ipynb)
*Or run locally from inside a Jove checkout.*

## Contents

0. [Setup](#setup)
1. [Video recap](#video)
2. [The toolkit at a glance](#toolkit)
3. [Alphabets and languages](#alphabet)
4. [Zero, Unit, and concatenation](#zero-unit)
5. [Exponentiation: $L^n$](#lexp)
6. [Star, truncated: $L^*_n$](#lstar)
7. [Reversal and homomorphism](#rev-homo)
8. [Powersets](#powset)
9. [Set operations and complement](#setops)
10. [Enumerating $\Sigma^*$: `nthnumeric`](#nthnumeric)
11. [Building bigger languages from simple ones](#building)
12. [Exercises](#exercises)
13. [Where next](#where-next)

<a id="setup"></a>
## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

<a id="video"></a>
## 1. Video recap

Under 18 minutes. Some of the code shown may differ from what is in Jove today
&mdash; for example, `nthnumeric` now takes a *list* as its second argument
&mdash; but the ideas are the same.

In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo('TAEYvJn5eGc')

<a id="toolkit"></a>
## 2. The toolkit at a glance

Every function below is defined, in full, later in this notebook; the same
definitions live in `jove.LangDef`, which the Setup cell imported. Each has a
docstring, so `help(lstar)` describes `lstar` fully.

| Function | Meaning | Section | Concept notebook |
|---|---|---|---|
| `lphi()` | $\emptyset$, the zero language | [4](#zero-unit) | |
| `lunit()` | $\{\varepsilon\}$, the unit language | [4](#zero-unit) | [Empty star is unit](../Concept-Empty-Star-Is-Unit/Concept-Empty-Star-Is-Unit.ipynb) |
| `lcat(L1,L2)` | $L_1 L_2$ | [4](#zero-unit) | |
| `lexp(L,n)` | $L^n$ | [5](#lexp) | |
| `lunion(L1,L2)` | $L_1 \cup L_2$ | [6](#lstar) | |
| `lstar(L,n)` | $L^*_n$, star truncated at $n$ | [6](#lstar) | [Three definitions](../Concept-Star-Three-Definitions/Concept-Star-Three-Definitions.ipynb), [Bounded star](../Concept-Lstar-Bounded/Concept-Lstar-Bounded.ipynb) |
| `srev(s)`, `lrev(L)` | reversal of a string / language | [7](#rev-homo) | [Reverse of a language](../Concept-Reverse-Of-A-Language/Concept-Reverse-Of-A-Language.ipynb) |
| `shomo(s,f)`, `lhomo(L,f)` | homomorphism of a string / language | [7](#rev-homo) | [Homomorphisms](../Concept-Homomorphisms/Concept-Homomorphisms.ipynb) |
| `powset(S)` | $2^S$, as a list of lists | [8](#powset) | |
| `lint(L1,L2)` | $L_1 \cap L_2$ | [9](#setops) | |
| `lsymdiff(L1,L2)` | $L_1 \oplus L_2$ | [9](#setops) | [Symmetric difference](../Concept-Symmetric-Difference-Subtraction/Concept-Symmetric-Difference-Subtraction.ipynb) |
| `lminus(L1,L2)` | $L_1 \setminus L_2$ | [9](#setops) | [Symmetric difference](../Concept-Symmetric-Difference-Subtraction/Concept-Symmetric-Difference-Subtraction.ipynb) |
| `lissubset`, `lissuperset` | $\subseteq$, $\supseteq$ | [9](#setops) | |
| `lcomplem(L,sigma,n)` | $\Sigma^*_n \setminus L$ | [9](#setops) | [Complementation](../Concept-Language-Complementation/Concept-Language-Complementation.ipynb) |
| `product(S1,S2)` | $S_1 \times S_2$ | [9](#setops) | |
| `nthnumeric(N,S)` | the $N$-th string of $S^*$ in numeric order | [10](#nthnumeric) | [nthnumeric](../Concept-Nthnumeric-Generator/Concept-Nthnumeric-Generator.ipynb) |

**Documentation conventions.** Docstrings start and end with `"""`. `In :` lists
each input as *concept (Python representation)*; `Out:` does the same for the
result; an example sometimes follows. One-line helpers get one-line docs.

In [ ]:
help(lstar)

<a id="alphabet"></a>
## 3. Alphabets and languages

> *You do not understand anything unless you have studied it in multiple ways.*
> &mdash; John McCarthy

Do the exercises on paper &mdash; that is the mathematics, and there is no
short-cut to it. Then play with the code here; that builds the intuition that
makes the mathematics stick.

**Alphabet.** An alphabet $\Sigma$ is a *finite, non-empty* set of symbols. They
are usually single characters, but they may be finite strings, or even (in a GNFA)
regular expressions.

**Language.** A language is a set of strings over $\Sigma$. In Jove a language is
almost always a Python **set of strings**; exceptions are pointed out where they
occur.

**Coding style.** The code could be object-oriented; it deliberately is not.
It uses as few Python constructs as possible, so that math and code sit at the
same level &mdash; in the spirit of Knuth's literate programs.

<a id="zero-unit"></a>
## 4. Zero, Unit, and concatenation

Concatenation plays the role of multiplication for languages:

$$L_1 L_2 \;=\; \{\,xy \;\mid\; x \in L_1 \,\wedge\, y \in L_2\,\}$$

Then

* **Zero** is $\emptyset$ (`lphi()`): like $3 \times 0 = 0$, $\;L\emptyset = \emptyset L = \emptyset$.
* **Unit** is $\{\varepsilon\}$ (`lunit()`): like $3 \times 1 = 3$, $\;L\{\varepsilon\} = \{\varepsilon\}L = L$.

In [ ]:
def lphi():
    """In : None.
       Out: Zero language, i.e. set({}).
    """
    return set({})  # {} alone would be a dict; so we write set(..)

def lunit():
    """In : None.
       Out: {""} (a language : a set).
    """
    return {""} # Set with epsilon

def lcat(L1,L2):
    """In : L1 (language : a set),
            L2 (language : a set).
       Out: L1 concat L2 (language : a set).
       Example:
       L1 = {'ab', 'bc'}
       L2 = {'11', 'ab', '22'}
       lcat(L1,L2) -> {'abab', 'bc22', 'ab11', 'ab22', 'bcab', 'bc11'}
    """
    return {x+y for x in L1 for y in L2}

In [ ]:
print("lphi()  =", lphi())
print("lunit() =", lunit(), "   <- one string, the empty one")
assert lphi() == set() and lunit() == {''} and len(lunit()) == 1

Python quotes strings with `'a'`, `"a"` or `"""a"""` interchangeably &mdash; the
choice only matters for nesting. Sets print in no particular order; compare
*contents*, not layout.

In [ ]:
L1 = {"a"*n for n in range(0,3)}     # {'', 'a', 'aa'}
L2 = {"b"*m for m in range(0,2)}     # {'', 'b'}
print("L1 =", L1)
print("L2 =", L2)
print("lcat(L1,L2) =", lcat(L1,L2))
assert lcat(L1,L2) == {'', 'a', 'aa', 'b', 'ab', 'aab'}

In [ ]:
L = {'a','bc'}
print("lcat(lphi(), L)  =", lcat(lphi(), L))
print("lcat(lunit(), L) =", lcat(lunit(), L))
assert lcat(lphi(), L)  == lcat(L, lphi())  == lphi()   # zero annihilates
assert lcat(lunit(), L) == lcat(L, lunit()) == L        # unit is identity

A second language, built by set comprehension:

$$M = \{\, 0^m 1^n \;\mid\; 0 \le m < 3,\; 0 \le n < 4,\; m < n \,\}$$

In [ ]:
M = {"0"*m + "1"*n for m in range(3) for n in range(4) if m < n}
print("M             =", M)
print("lcat(L,M)     =", lcat(L,M))
print("lcat(M,lphi()) =", lcat(M,lphi()))
print("lcat(M,lunit()) =", lcat(M,lunit()))
assert lcat(M,lphi()) == lphi() and lcat(M,lunit()) == M

### Concatenation vs. Cartesian product

Concatenation draws $x$ from $L_1$ and $y$ from $L_2$ just as $L_1 \times L_2$
does &mdash; but **glues** them instead of **pairing** them. Gluing can make two
different pairs collide into one string, so $|L_1L_2| \le |L_1 \times L_2|$.

In [ ]:
x = {'', 'a'}
xcrossx = {(a,b) for a in x for b in x}
print("x x  =", lcat(x,x))
print("x × x =", xcrossx)
print("sizes:", len(lcat(x,x)), "vs", len(xcrossx), "  ('','a') and ('a','') both glue to 'a'")
assert len(lcat(x,x)) < len(xcrossx)

**Try it.**

1. Find non-empty $L_1, L_2$ with $|L_1 L_2| = |L_1| \cdot |L_2|$, and others where it is smaller.
2. Is `lcat` commutative? Find $L_1, L_2$ with $L_1L_2 \ne L_2L_1$.
3. Is it associative? Test `lcat(lcat(A,B),C) == lcat(A,lcat(B,C))` on a few examples, then prove it.

<a id="lexp"></a>
## 5. Exponentiation: $L^n$

Exponentiation is repeated multiplication &mdash; here, repeated concatenation:

$$L^0 = \{\varepsilon\}, \qquad L^n = L\,L^{n-1}$$

$L^0 = \{\varepsilon\}$ is the only sensible base case.
**Question:** if you set $L^0 = \emptyset$ instead, what is $L^n$ for every $n$?

In [ ]:
def lexp(L,n):
    """In : L (language : a set),
            n (exponent : a nat).
       Out: L^n (language : a set).
       Example:
       L = {'ab', 'bc'}
       n = 2
       lexp(L,2) -> {'abab', 'bcab', 'bcbc', 'abbc'}
    """
    return lunit() if n == 0 else lcat(L, lexp(L, n-1))

In [ ]:
L = {'a','bc'}
print("lexp(L,0) =", lexp(L,0))
print("lexp(L,2) =", lexp(L,2))
assert lexp(L,0) == lunit()
assert lexp(L,2) == {'aa', 'abc', 'bca', 'bcbc'}

In [ ]:
print("M         =", M)
print("lexp(M,2) =", lexp(M,2))
assert lexp(lcat(L,M),1) == lcat(L,M)          # L^1 = L

**Try it.**

1. Predict `len(lexp({'a','bc'}, 5))`, then check. Why is there no collision?
2. Find a two-string $L$ for which `len(lexp(L,3))` is *less* than $2^3$.
3. What are `lexp(lphi(), 0)` and `lexp(lphi(), 3)`? Explain the difference.

<a id="lstar"></a>
## 6. Star, truncated: $L^*_n$

With `lexp` in hand we define **star up to $n$**:

$$L^*_0 = \{\varepsilon\}, \qquad L^*_n = L^n \cup L^*_{n-1}$$

The real star is $L^* = L^*_\infty$, which we will not try to "run" in Python.
Star yields an infinite set in almost every case, so Jove only ever computes the
truncated version.

In [ ]:
def lunion(L1,L2):
    """In : L1 (language : a set),
            L2 (language : a set).
       Out: L1 union L2 (language : a set).
    """
    return L1 | L2

def lstar(L,n):
    """In : L (language : a set),
            n (bound for lstar : a nat).
       Out: L*_n (language : a set)
       Example:
       L = {'ab','bc'}
       n = 2
       lstar(L,2) -> {'abab', 'bcbc', 'ab', 'abbc', '', 'bc', 'bcab'}
    """
    return lunit() if n == 0 else lunion(lexp(L,n), lstar(L,n-1))

**Try it.** Change the bound below from 2 to 5 and note how many strings `lstar`
returns. Can you predict the count before running?

In [ ]:
L1 = {'a','bc'}
print(len(lstar(L1,2)), "strings:", lstar(L1,2))
assert lstar(L1,2) == {'', 'a', 'bc', 'aa', 'abc', 'bca', 'bcbc'}

In [ ]:
L2 = {'ab','bc'}
print("lstar(L2,2) =", lstar(L2,2))
print("lstar(L2,3) has", len(lstar(L2,3)), "strings")
assert lstar(L2,2) == {'', 'ab', 'bc', 'abab', 'abbc', 'bcab', 'bcbc'}

### The star of an alphabet is the universe of strings

View an alphabet as a language of length-1 strings and star it: $\Sigma^*$ is
*every* string over $\Sigma$. Truncated at $n$, that is every string of length
$\le n$ &mdash; $2^0 + 2^1 + \dots + 2^n$ of them for a two-letter alphabet.

<!-- nav-strip -->

---

&larr;&nbsp;[Ch2&nbsp;17.&nbsp;The Language Operations, Animated](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter2-Lang/Concept-Language-Operations-Animated/Concept-Language-Operations-Animated.ipynb) &nbsp;&middot;&nbsp; [**Chapter 3** index](https://github.com/ganeshutah/Jove/blob/master/Chapter3-Star/README.md) &nbsp;&middot;&nbsp; [Ch3&nbsp;1.&nbsp;Star: Three Equivalent Definitions](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter3-Star/Concept-Star-Three-Definitions/Concept-Star-Three-Definitions.ipynb)&nbsp;&rarr;

---

In [ ]:
Sig01 = {'0','1'}
for n in range(4):
    print("lstar(Sig01,%d) =" % n, sorted(lstar(Sig01,n), key=lambda s: (len(s), s)))
    assert len(lstar(Sig01,n)) == 2**(n+1) - 1

### Explore with sliders

Pick a language from the drop-down and slide the truncation limit. If the slider
were infinite and you slid it all the way right, you would be watching the true
star. Try `lphi` and `lunit` especially &mdash; and compare `{'', 'bc'}` with
`{'a', 'bc'}`.

In [ ]:
import ipywidgets as wdg

Langs = [('{a,bc}',    {'a','bc'}),
         ('{,bc}',     {'','bc'}),
         ('{ab,bc}',   {'ab','bc'}),
         ('{a,b,c}',   {'a','b','c'}),
         ('M',         {'011', '111', '11', '0111', '00111', '1'}),
         ('Sigma={a,b}', {'a','b'}),
         ('Sigma={0,1}', {'0','1'}),
         ('lphi',      lphi()),
         ('lunit',     lunit())]

wdg.interact(lambda L, n: sorted(lstar(L,n), key=lambda s: (len(s), s)),
             L=Langs, n=(0,7));

Starring never escapes $\{\varepsilon\}$ when you start there, and starring twice
gains nothing:

In [ ]:
print("lstar(lunit(),10)            =", lstar(lunit(),10))
print("lstar(lstar(lunit(),10),10)  =", lstar(lstar(lunit(),10),10))
print("lstar(lphi(),10)             =", lstar(lphi(),10))
assert lstar(lunit(),10) == lstar(lstar(lunit(),10),10) == lstar(lphi(),10) == lunit()

**Go deeper:**
[Three definitions of star](../Concept-Star-Three-Definitions/Concept-Star-Three-Definitions.ipynb) &middot;
[$\Sigma^*$, the universal language](../Concept-Sigma-Star-Universal-Language/Concept-Sigma-Star-Universal-Language.ipynb) &middot;
[Bounded star `lstar`](../Concept-Lstar-Bounded/Concept-Lstar-Bounded.ipynb) &middot;
[$\emptyset^* = \{\varepsilon\}$](../Concept-Empty-Star-Is-Unit/Concept-Empty-Star-Is-Unit.ipynb) &middot;
[Star generates all strings](../Concept-Star-Generates-All-Strings/Concept-Star-Generates-All-Strings.ipynb) &middot;
[Reasoning about star](../Concept-Reasoning-About-Star/Concept-Reasoning-About-Star.ipynb) &middot;
[$L^{**} = L^*$](../Concept-Star-Star-Is-Star/Concept-Star-Star-Is-Star.ipynb)

<a id="rev-homo"></a>
## 7. Reversal and homomorphism

* `srev` reverses a string; `lrev` maps `srev` over a language.
* `shomo` applies a character map `f` to every character of a string; `lhomo`
  maps that over a language. With `f` a rotation you can "rot13" your email.

Python has no built-in string reverse; the slice `S[::-1]` (start and stop left
empty, stride $-1$) is the usual idiom. `"".join(reversed(S))` also works.

In [ ]:
def srev(S):
    """In : S (string)
       Out: reverse of S (string)
       Example:
       srev('ab') -> 'ba'
    """
    return S[::-1]

def lrev(L):
    """In : L (language : a set)
       Out: reverse of L (language : a set)
       Example:
       lrev({'ab', 'bc'}) -> {'cb', 'ba'}
    """
    return set(map(lambda x: srev(x), L))

def shomo(S,f):
    """In : S (string)
            f (function from char to char)
       Out: String homomorphism of S wrt f.
       Example:
       S = "abcd"
       f = lambda x: chr( (ord(x)+1) % 256 )
       shomo("abcd",f) -> 'bcde'
    """
    return "".join(map(f,S))

def lhomo(L,f):
    """In : L (language : set of strings)
            f (function from char to char)
       Out: Lang. homomorphism of L wrt f (language : set of str)
       Example:
       L = {"Hello there", "a", "A"}
       f = rot13 = lambda x: chr( (ord(x)+13) % 256 )
       lhomo(L, rot13) -> {'N', 'Uryy|-\x81ur\x7fr', 'n'}
    """
    return set(map(lambda S: shomo(S,f), L))

In [ ]:
L = {'ab', '007'}
rot1 = lambda x: chr((ord(x)+1) % 256)     # rotate every character by one

print('lrev(L)              =', lrev(L))
print('lhomo(L, rot1)       =', lhomo(L, rot1))
print('lrev(lhomo(L, rot1)) =', lrev(lhomo(L, rot1)))
print('lhomo(lrev(L), rot1) =', lhomo(lrev(L), rot1), '  <- same: the two commute')
assert lrev(L) == {'ba', '700'}
assert lhomo(L, rot1) == {'bc', '118'}
assert lrev(lhomo(L, rot1)) == lhomo(lrev(L), rot1)

**Go deeper:**
[Reverse of a language](../Concept-Reverse-Of-A-Language/Concept-Reverse-Of-A-Language.ipynb) &middot;
[Homomorphisms](../Concept-Homomorphisms/Concept-Homomorphisms.ipynb) &middot;
[The crux of homomorphisms](../Concept-Crux-Of-Homomorphisms/Concept-Crux-Of-Homomorphisms.ipynb)

<a id="powset"></a>
## 8. Powersets

Needed again for NFAs (Chapter 7), where a DFA state is a *set* of NFA states.
Python sets cannot hold sets (they are unhashable), so `powset` returns a list of
lists. The recursion: take some $x \in S$; then

$$2^S \;=\; 2^{S \setminus \{x\}} \;\cup\; \{\, Y \cup \{x\} \;\mid\; Y \in 2^{S \setminus \{x\}} \,\}$$

&mdash; every subset either leaves $x$ out or puts it back in.

In [ ]:
def powset(S):
    """In : S (set)
       Out: List of lists representing powerset.
            Since sets/lists are unhashable, we convert the set
            to a list, perform the powerset operations, leaving
            the result as a list (can't convert back to a set).
       Example:
       S = {'ab', 'bc'}
       powset(S) -> [['ab', 'bc'], ['bc'], ['ab'], []]
    """
    L = list(S)
    if L == []:
        return [[]]
    else:
        pow_rest0 = powset(L[1:])
        pow_rest1 = list(map(lambda Ls: [L[0]] + Ls, pow_rest0))
        return pow_rest0 + pow_rest1

In [ ]:
P = powset({'a','b','c'})
print(P)
assert len(P) == 2**3
assert sorted(map(sorted, P)) == sorted(map(sorted,
       [[], ['a'], ['b'], ['c'], ['a','b'], ['a','c'], ['b','c'], ['a','b','c']]))

<a id="setops"></a>
## 9. Set operations and complement

The familiar set operations, with language names:
`lunion`, `lint`, `lsymdiff`, `lminus`, `lissubset`, `lissuperset`, and
`product` (Cartesian product &mdash; compare with concatenation in
[Section 4](#zero-unit)).

**Pay special attention to `lcomplem`.** The complement of $L$ is taken relative
to the universe $\Sigma^*$. We can only compute finite approximations of star, so
`lcomplem` takes a truncation bound too: it returns $\Sigma^*_n \setminus L$.
Follow the experiments below, then picture what happens as $n \to \infty$.

In [ ]:
def lint(L1,L2):
    """In : L1 (language : set of strings)
            L2 (language : set of strings)
       Out: L1 intersection L2 (sets of strings)
    """
    return L1 & L2

def lsymdiff(L1,L2):
    """In : L1 (language : set of strings)
            L2 (language : set of strings)
       Out: (L1 - L2) union (L2 - L1) (sets of strings)
       Example:
       lsymdiff({'ab', 'bc'}, {'11', 'ab', '22'}) -> {'11', '22', 'bc'}
    """
    return L1 ^ L2

def lminus(L1,L2):
    """Language subtraction of two languages (sets of strings)
       Can do it as L1.difference(L2) also.
    """
    return L1 - L2

def lissubset(L1,L2):
    """In : L1 (language : set of strings)
            L2 (language : set of strings)
       Out: L1 is subset or equal to L2 (True/False)
    """
    return L1 <= L2

def lissuperset(L1,L2):
    """In : L1 (language : set of strings)
            L2 (language : set of strings)
       Out: L1 is superset or equal to L2 (True/False)
    """
    return L1 >= L2

def lcomplem(L,sigma,n):
    """In : L (language : set of strings)
            sigma (alphabet : set of strings)
            n (finite limit for lstar : int)
       Out : sigma*_n - L (language : set of strings)
       Example:
       L = {'0', '10', '010'}
       sigma = {'0', '1'}
       n = 3
       lcomplem(L,sigma,n) ->
       {'', '000', '101', '011', '00', '1',
        '001', '110', '111', '100', '01', '11'}
    """
    return lstar(sigma,n) - L

def product(S1,S2):
    """In : S1 (set)
            S2 (set)
       Out: Cartesian product of S1 and S2 (set of pairs)
    """
    return {(x,y) for x in S1 for y in S2}

In [ ]:
A = {'ab', 'bc'}
B = {'11', 'ab', '22'}
print("lint(A,B)     =", lint(A,B))
print("lsymdiff(A,B) =", lsymdiff(A,B))
print("lminus(A,B)   =", lminus(A,B))
print("product(A,B)  =", product(A,B))
assert lsymdiff(A,B) == lunion(lminus(A,B), lminus(B,A)) == {'11', '22', 'bc'}
assert lissubset(lint(A,B), A) and lissuperset(lunion(A,B), B)

Building languages out of others: $L_3 = L_1 (\{\varepsilon\} \cup L_2 L_1)$ where
$L_1 = \{0101\}$ and $L_2 = \{0,1\}^*_2$.

In [ ]:
L1 = {'0101'}
L2 = lstar({'0','1'}, 2)
L2L1 = lcat(L2, L1)                  # concat of L2 and L1
L3 = lcat(L1, lunion(lunit(), L2L1))
print("L2L1 =", L2L1)
print("L3   =", L3)
assert L1 <= L3 and len(L3) == 1 + len(L2L1)

In [ ]:
L4 = {'0', '10', '010'}
print('L4 =', L4)
for n in range(1, 5):
    C = lcomplem(L4, {'0','1'}, n)
    print("---- lcomplem(L4, {'0','1'}, %d)  (%d strings)" % (n, len(C)))
    print(sorted(C, key=lambda s: (len(s), s)))
    assert lint(C, L4) == lphi()
    assert lunion(C, L4) >= lstar({'0','1'}, n)
print("In the limit: everything in {0,1}* except '0', '10' and '010'.")

**Go deeper:**
[Language complementation](../Concept-Language-Complementation/Concept-Language-Complementation.ipynb) &middot;
[Symmetric difference and subtraction](../Concept-Symmetric-Difference-Subtraction/Concept-Symmetric-Difference-Subtraction.ipynb)

<a id="nthnumeric"></a>
## 10. Enumerating $\Sigma^*$: `nthnumeric`

`lstar` gives finite *slices* of $\Sigma^*$. To list $\Sigma^*$ **one string at a
time, missing none**, use *numeric order*: shorter strings first, and strings of
equal length in dictionary order. `nthnumeric(N, S)` returns the $N$-th string of
$S^*$ in that order, for a two-letter alphabet given as a **list** (so the order
of the two letters is fixed).

In [ ]:
from math import floor, log, pow

def nthnumeric(N, S=['a','b']):
    """Assume S ('Sigma') is a 2-sized list of chars (default ['a','b']).
       Produce the Nth string in numeric order, where N >= 0.
       Idea : Given N, get b = floor(log_2(N+1)) - need that
       many places; what to fill in the places is the binary
       code for N - (2^b - 1) with 0 as S[0] and 1 as S[1].
    """
    assert(type(S)==list # This makes sure that the order does not flip
          ),"nthnumeric now expects to be given a two-element list!!"
    assert(len(S)==2
          ),"Expected to be given an S of length 2."
    if(N==0):
        return ''
    else:
        width = floor(log(N+1, 2))
        tofill = int(N - pow(2, width) + 1)
        relevant_binstr = bin(tofill)[2::] # strip the leading 0b
        len_to_makeup = width - len(relevant_binstr)
        return (S[0]*len_to_makeup +
                shomo(relevant_binstr,
                      lambda x: S[1] if x=='1' else S[0]))

In [ ]:
first15 = [nthnumeric(i, ['a','b']) for i in range(15)]
print(first15)
assert set(first15) == lstar({'a','b'}, 3)     # the first 2^4-1 cover {a,b}*_3

**Go deeper:**
[Lexicographic order, and why it fails to enumerate](../Concept-Lexicographic-Order/Concept-Lexicographic-Order.ipynb) &middot;
[Numeric order](../Concept-Numeric-Order/Concept-Numeric-Order.ipynb) &middot;
[`lexlt`](../Concept-Lexlt-Predicate/Concept-Lexlt-Predicate.ipynb) &middot;
[`nthnumeric`](../Concept-Nthnumeric-Generator/Concept-Nthnumeric-Generator.ipynb)

<a id="building"></a>
## 11. Building bigger languages from simple ones

Finite slices of $a^*$, $b^*$, $c^*$, and of the "matched" languages
$\{a^ib^i\}$, $\{b^ic^i\}$ and $\{a^ib^ic^i\}$ (all for $i < 8$):

In [ ]:
As   = {"a"*i for i in range(8)}
Bs   = {"b"*i for i in range(8)}
Cs   = {"c"*i for i in range(8)}
ABs  = {"a"*i + "b"*i for i in range(8)}
BCs  = {"b"*i + "c"*i for i in range(8)}
ABCs = {"a"*i + "b"*i + "c"*i for i in range(8)}
for name in ['As', 'ABs', 'BCs', 'ABCs']:
    print("%-5s =" % name, sorted(eval(name), key=len))

Concatenating $a^*$ with $b^*$ gives $a^*b^*$ &mdash; the counts are chosen
**independently**, so it is much bigger than $\{a^ib^i\}$. Likewise
$\{a^ib^i\}\,c^*$ ties $a$ to $b$ but leaves $c$ free. No amount of concatenating
these pieces yields $\{a^ib^ic^i\}$, where all three counts are tied together.
Chapters 4 and 11 explain why that matters.

In [ ]:
print("|lcat(As,Bs)|  =", len(lcat(As,Bs)), "   vs |ABs|  =", len(ABs))
print("|lcat(ABs,Cs)| =", len(lcat(ABs,Cs)), "   vs |ABCs| =", len(ABCs))
assert ABs < lcat(As,Bs)
assert ABCs < lcat(ABs,Cs) and ABCs < lcat(As,BCs)
assert 'aabbc' in lcat(ABs,Cs) and 'aabbc' not in ABCs

<a id="exercises"></a>
## 12. Exercises

Now do the exercises at the end of Chapters 2 and 3 of the book &mdash; on paper
first, then check what you can here. A few to start with:

1. What is `lstar({'', 'bc'}, n)`? Compare it with `lstar({'bc'}, n)` and explain.
2. Prove $\emptyset^* = \{\varepsilon\}$ from the definition of $L^*_n$, then check it with `lstar`.
3. Show $\overline{L_1 \cup L_2} = \overline{L_1} \cap \overline{L_2}$ using `lcomplem` with the *same* bound on both sides. What goes wrong if the bounds differ?
4. Is $\mathrm{rev}(L_1L_2) = \mathrm{rev}(L_1)\,\mathrm{rev}(L_2)$? If not, fix the right-hand side.
5. Write `lpow_check(L, n)` that asserts `lexp(L, n) <= lstar(L, n)` for several languages.

In [ ]:
# Your work here.

<a id="where-next"></a>
## 13. Where next

The Chapter 3 concept notebooks, in reading order:

| # | Concept |
|---|---|
| 1 | [Star: Three Equivalent Definitions](../Concept-Star-Three-Definitions/Concept-Star-Three-Definitions.ipynb) |
| 2 | [The Two Roles of Star: Repetition, and the Universal Language $\Sigma^*$](../Concept-Sigma-Star-Universal-Language/Concept-Sigma-Star-Universal-Language.ipynb) |
| 3 | [Star Bounded at $n$ ($L^*_n$) and `lstar`](../Concept-Lstar-Bounded/Concept-Lstar-Bounded.ipynb) |
| 4 | [$\emptyset^*=\{\varepsilon\}$, and $\varepsilon\in L^*$ for Every $L$](../Concept-Empty-Star-Is-Unit/Concept-Empty-Star-Is-Unit.ipynb) |
| 5 | [Star Generates All Strings Over an Alphabet](../Concept-Star-Generates-All-Strings/Concept-Star-Generates-All-Strings.ipynb) |
| 6 | [Reasoning About Star](../Concept-Reasoning-About-Star/Concept-Reasoning-About-Star.ipynb) |
| 7 | [Language Complementation, Relative to $\Sigma^*$](../Concept-Language-Complementation/Concept-Language-Complementation.ipynb) |
| 8 | [Symmetric Difference and Language Subtraction](../Concept-Symmetric-Difference-Subtraction/Concept-Symmetric-Difference-Subtraction.ipynb) |
| 9 | [Reverse of a Language](../Concept-Reverse-Of-A-Language/Concept-Reverse-Of-A-Language.ipynb) |
| 10 | [String and Language Homomorphisms](../Concept-Homomorphisms/Concept-Homomorphisms.ipynb) |
| 11 | [The Crux of Homomorphisms](../Concept-Crux-Of-Homomorphisms/Concept-Crux-Of-Homomorphisms.ipynb) |
| 12 | [Theorem: $L^{**}=L^*$](../Concept-Star-Star-Is-Star/Concept-Star-Star-Is-Star.ipynb) |
| 13 | [Lexicographic Order, and Why It Fails to Enumerate](../Concept-Lexicographic-Order/Concept-Lexicographic-Order.ipynb) |
| 14 | [Numeric Order as a Genuine Enumeration](../Concept-Numeric-Order/Concept-Numeric-Order.ipynb) |
| 15 | [Formal Definition of Lexicographic Order, and `lexlt`](../Concept-Lexlt-Predicate/Concept-Lexlt-Predicate.ipynb) |
| 16 | [`nthnumeric`: Coding a Numeric-Order Generator](../Concept-Nthnumeric-Generator/Concept-Nthnumeric-Generator.ipynb) |

Or search every concept in every chapter:

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter3-Star/Concept-START-HERE')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')